In [1]:
from algorithms.raw.reader import read_raw_sensor_data
from algorithms.demosaicing._malvar_he_culter import malvar_he_cutler_demosaicing
from algorithms.white_balance._camera_white_balance import camera_white_balance
from algorithms.hue_saturation_map._hue_saturation_map import apply_hue_sat_map
from algorithms.gc.iec_gamma_correction import iec_gamma_correction
from algorithms.export._jpeg_export import export_jpeg
from algorithms.tools._lut_tools import read_hue_sat_lut_from_dcp
from algorithms.black_light_subtraction._black_light_subtraction import linearize_raw

def main(raw_img_path: str, dcp_file_path: str, output_img_path: str): 
    raw_img, bayer_pattern, black_levels, white_level, wb_gains = read_raw_sensor_data(raw_img_path)
    raw_img = linearize_raw(raw_img, bayer_pattern, black_levels, white_level)
    demosaiced_img = malvar_he_cutler_demosaicing(raw_img)
    white_balanced_img = camera_white_balance(demosaiced_img, wb_gains)
    res = read_hue_sat_lut_from_dcp(dcp_file_path)
    if res is None : return
    (color_matrix_1, color_matrix_2, forward_matrix_1, forward_matrix_2, low_temp_lut, high_temp_lut, calib_illum_1, calib_illum_2) = res
    hue_sat_corrected_img = apply_hue_sat_map(
        white_balanced_img,
        wb_gains,
        color_matrix_1,
        color_matrix_2,
        forward_matrix_1,
        forward_matrix_2,
        low_temp_lut,
        high_temp_lut,
        calib_illum_1,
        calib_illum_2,
    )
    gamma_corrected_img = iec_gamma_correction(hue_sat_corrected_img)
    export_jpeg(gamma_corrected_img, output_img_path)

In [ ]:
raw_path = "/home/amayas/telecom-paris/1a/workspace/artishow/raw/FEA06583.ARW"
dcp_path = "/home/amayas/telecom-paris/1a/workspace/artishow/dcp-files/SONY ILCE-7RM3.dcp"
out_path = "/home/amayas/Téléchargements/output.jpeg"

%prun main(raw_path, dcp_path, out_path)

<tifffile.TiffPage 0 @8> missing data offset tag
<tifffile.TiffPage 0 @8> missing data ByteCounts tag


TiffTag 50708 UniqueCameraModel @10 ASCII[15] @242 = Sony ILCE-7RM3
TiffTag 50721 ColorMatrix1 @22 SRATIONAL[9] @258 = (7683, 10000, -3276, 10000, 
TiffTag 50722 ColorMatrix2 @34 SRATIONAL[9] @330 = (6640, 10000, -1847, 10000, 
TiffTag 50778 CalibrationIlluminant1 @46 SHORT @54 = 17
TiffTag 50779 CalibrationIlluminant2 @58 SHORT @66 = 21
TiffTag 50932 ProfileCalibrationSignature @70 ASCII[10] @402 = com.adobe
TiffTag 50936 ProfileName @82 ASCII[15] @414 = SONY ILCE-7RM3
TiffTag 50937 ProfileHueSatMapDims @94 LONG[3] @430 = (90, 30, 1)
TiffTag 50938 ProfileHueSatMapData1 @106 FLOAT[8100] @442 = array([0. , 1. , 1.
TiffTag 50939 ProfileHueSatMapData2 @118 FLOAT[8100] @32842 = array([0. , 1. , 
TiffTag 50940 ProfileToneCurve @130 FLOAT[16384] @65242 = array([0.0000000e+00,
TiffTag 50941 ProfileEmbedPolicy @142 LONG @150 = 3
TiffTag 50942 ProfileCopyright @154 ASCII[24] @130778 = Copyright Maciej Dworak
TiffTag 50964 ForwardMatrix1 @166 SRATIONAL[9] @130802 = (6668, 10000, 1299, 10
TiffTag